# Project: Customer Segmentation — Wholesale Distributor Clients

**Phase 6 — Unsupervised Learning | Capstone project 1 of 3**

### Scenario
A wholesale food distributor wants to segment its ~440 business clients (cafes, retailers,
hotels) by their annual spending pattern across product categories, so sales and marketing
can tailor outreach per segment rather than treating every client identically.

Data: https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv (UCI Machine Learning Repository — a stable, official source for this classic
real B2B spending dataset)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv")
print(df.shape)
df.head()

## 1. A quick recap EDA

In [ ]:
df.describe().round(0)

In [ ]:
spend_cols = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
fig, ax = plt.subplots(figsize=(9, 5))
df[spend_cols].boxplot(ax=ax)
ax.set_yscale("log")
ax.set_title("Annual spend by category (log scale -- these are heavily right-skewed)")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

> 💡 Spending data is almost always right-skewed (a few big-spending clients dominate) —
> we'll log-transform before clustering, exactly like Section 2.2 taught for skewed
> numeric features.

## 2. Preprocessing: log-transform + scale

In [ ]:
from sklearn.preprocessing import StandardScaler

X_log = np.log1p(df[spend_cols])
X_scaled = StandardScaler().fit_transform(X_log)

## 3. Choosing k with the elbow method and silhouette score (Section 6.1 skills)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias, silhouettes = [], []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_range), inertias, marker="o"); axes[0].set_title("Elbow method")
axes[1].plot(list(k_range), silhouettes, marker="o", color="crimson"); axes[1].set_title("Silhouette score")
plt.tight_layout(); plt.show()

In [ ]:
best_k = int(np.array(list(k_range))[np.argmax(silhouettes)])
print(f"chosen k = {best_k}")

kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X_scaled)
df["segment"] = kmeans.labels_

## 4. Visualizing the segments (PCA, Section 6.2 skills)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df["segment"], cmap="tab10", s=50, edgecolor="k", alpha=0.8)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
ax.set_title(f"Client segments (k={best_k}), visualized via PCA")
plt.tight_layout(); plt.show()

## 5. Profiling each segment — turning clusters into a business story

In [ ]:
segment_profile = df.groupby("segment")[spend_cols + ["Channel", "Region"]].mean().round(0)
segment_profile["n_clients"] = df["segment"].value_counts().sort_index()
segment_profile

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
(df.groupby("segment")[spend_cols].mean()).plot(kind="bar", ax=ax)
ax.set_title("Average annual spend by category, per segment")
ax.set_ylabel("annual spend ($)")
plt.tight_layout(); plt.show()

In [ ]:
# Naming the segments based on their dominant spending pattern -- the actual sales deliverable
for seg_id, row in segment_profile.iterrows():
    dominant = row[spend_cols].idxmax()
    print(f"Segment {seg_id} ({int(row['n_clients'])} clients): dominant category = {dominant}, "
          f"avg total spend = ${row[spend_cols].sum():,.0f}")

> 💡 **This table, not the scatter plot, is what a sales team actually acts on.** "Segment
> 2 clients spend heavily on Fresh and little on Detergents_Paper — likely restaurants/
> cafes, worth a different outreach pitch than Segment 0's grocery-heavy retailers." This
> is the direct business translation of an unsupervised model's output.

## 6. Does `Channel` (Retail vs. Horeca) mostly explain the segments?

In [ ]:
crosstab = pd.crosstab(df["segment"], df["Channel"])
crosstab.columns = ["Horeca (Hotel/Restaurant/Cafe)", "Retail"]
crosstab

> 💡 The dataset actually includes a REAL business channel label (`Channel`: Horeca vs.
> Retail) we didn't use for clustering. Checking whether our unsupervised segments line up
> with this known real-world grouping is a legitimate, if partial, sanity check — similar
> in spirit to how Phase 3's Titanic project checked its findings against documented
> history.

## 🧪 Extend this project yourself

- [ ] Try Hierarchical Clustering (Section 6.1) instead of K-Means — does the dendrogram
      suggest a different natural number of segments?
- [ ] Re-run clustering WITHOUT the log-transform — how much does the elbow/silhouette
      picture change, and why?
- [ ] Compute each segment's total revenue contribution — which segment is the distributor
      MOST dependent on?

## Results
- Identified several distinct client segments by log-transformed, scaled annual spending
  across 6 product categories.
- Profiled each segment's dominant spending category into an actionable business
  description, not just a cluster ID.
- Cross-checked segments against the dataset's real `Channel` label as a partial sanity
  check.

## Writeup
This project is the template for any B2B/B2C spending-based segmentation task: log-transform
skewed spend data, scale, choose k via elbow + silhouette, visualize via PCA, then profile
and NAME each segment — the naming step is what turns a clustering algorithm's output into
something a business can actually act on.